In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_4.py ──
"""
# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 4: Shared Utilities
# ════════════════════════════════════════════════════════════════════════
#
# Common infrastructure for all Exercise 4 technique files:
#   - AG News loading + caching (120K train, 7.6K test)
#   - Vocabulary building and text-to-index conversion
#   - ExperimentTracker + ModelRegistry + OnnxBridge setup
#   - Shared constants, device detection, visualisation helpers
#
# ════════════════════════════════════════════════════════════════════════
"""

import asyncio
import math
from collections import Counter
from pathlib import Path
import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, TensorDataset

import plotly.graph_objects as go

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml import OnnxBridge

# ════════════════════════════════════════════════════════════════════════
# Environment + Device
# ════════════════════════════════════════════════════════════════════════
setup_environment()

torch.manual_seed(42)
np.random.seed(42)
DEVICE = get_device()

# ════════════════════════════════════════════════════════════════════════
# Constants
# ════════════════════════════════════════════════════════════════════════
CLASS_NAMES = ["World", "Sports", "Business", "Sci/Tech"]
MAX_LEN = 40
VOCAB_SIZE = 15000
EPOCHS_SCRATCH = 8
BERT_MODEL_NAME = "bert-base-uncased"
BERT_MAX_LEN = 64
BERT_EPOCHS = 3
BERT_LR = 2e-5
BERT_BATCH_SIZE = 32


# ════════════════════════════════════════════════════════════════════════
# Core Attention Function (shared across 01, 02, 05)
# ════════════════════════════════════════════════════════════════════════
def scaled_dot_product_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    mask: torch.Tensor | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute scaled dot-product attention.

    Attention(Q, K, V) = softmax( Q K^T / sqrt(d_k) ) V

    Args:
        q: Query tensor of shape (B, L_q, d_k)
        k: Key tensor of shape (B, L_k, d_k)
        v: Value tensor of shape (B, L_k, d_v)
        mask: Optional mask of shape (B, L_q, L_k). Positions with 0 are
              masked out (set to -inf before softmax).

    Returns:
        (output, attention_weights) where output has shape (B, L_q, d_v)
        and attention_weights has shape (B, L_q, L_k).
    """
    d_k = q.size(-1)
    scores = torch.einsum("bqd,bkd->bqk", q, k) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    out = torch.einsum("bqk,bkd->bqd", weights, v)
    return out, weights


# ════════════════════════════════════════════════════════════════════════
# AG News Loading
# ════════════════════════════════════════════════════════════════════════
def load_ag_news_split(split: str, cache_name: str) -> pl.DataFrame:
    """Load AG News split, caching to parquet for subsequent runs."""
    cache = Path.cwd() / "data" / "mlfp05" / cache_name
    cache.parent.mkdir(parents=True, exist_ok=True)
    if cache.exists():
        return pl.read_parquet(cache)
    print(f"  Downloading AG News {split} from HuggingFace...")
    ds = load_dataset("fancyzhx/ag_news", split=split)
    df = pl.from_pandas(ds.to_pandas())
    df.write_parquet(cache)
    return df


def load_ag_news() -> tuple[pl.DataFrame, pl.DataFrame]:
    """Load full AG News train + test splits with status reporting."""
    print("\n== Loading FULL AG News (120K train + 7.6K test) ==")
    train_df = load_ag_news_split("train", "ag_news_full_train.parquet")
    test_df = load_ag_news_split("test", "ag_news_full_test.parquet")
    print(f"  train rows: {len(train_df):,}   test rows: {len(test_df):,}")
    print(f"  sample headline: {train_df['text'][0][:80]!r}")
    print(f"  class balance (train): {dict(Counter(train_df['label'].to_list()))}")
    return train_df, test_df


# ════════════════════════════════════════════════════════════════════════
# Vocabulary + Tokenisation (for from-scratch models: LSTM + Transformer)
# ════════════════════════════════════════════════════════════════════════
def build_vocab(texts: list[str], max_vocab: int = VOCAB_SIZE) -> dict[str, int]:
    """Build a word-level vocabulary from text list."""
    words: Counter[str] = Counter()
    for t in texts:
        words.update(t.lower().split())
    vocab = {"<pad>": 0, "<unk>": 1}
    for word, _ in words.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    return vocab


def text_to_indices(
    text: str, vocab: dict[str, int], max_len: int = MAX_LEN
) -> list[int]:
    """Convert text to padded index sequence."""
    tokens = text.lower().split()[:max_len]
    idxs = [vocab.get(t, 1) for t in tokens]
    return (idxs + [0] * (max_len - len(idxs)))[:max_len]


def prepare_dataloaders(
    train_df: pl.DataFrame,
    test_df: pl.DataFrame,
    vocab: dict[str, int],
    batch_size: int = 128,
) -> tuple[
    DataLoader, DataLoader, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor
]:
    """Tokenise AG News and build DataLoaders for from-scratch models.

    Returns: (train_loader, val_loader, train_t, train_y, test_t, test_y)
    """
    train_tokens = np.array(
        [text_to_indices(t, vocab, MAX_LEN) for t in train_df["text"].to_list()],
        dtype=np.int64,
    )
    train_labels = np.array(train_df["label"].to_list(), dtype=np.int64)
    test_tokens = np.array(
        [text_to_indices(t, vocab, MAX_LEN) for t in test_df["text"].to_list()],
        dtype=np.int64,
    )
    test_labels = np.array(test_df["label"].to_list(), dtype=np.int64)

    train_t = torch.from_numpy(train_tokens).to(DEVICE)
    train_y = torch.from_numpy(train_labels).to(DEVICE)
    test_t = torch.from_numpy(test_tokens).to(DEVICE)
    test_y = torch.from_numpy(test_labels).to(DEVICE)

    train_loader = DataLoader(
        TensorDataset(train_t, train_y), batch_size=batch_size, shuffle=True
    )
    val_loader = DataLoader(TensorDataset(test_t, test_y), batch_size=batch_size)

    return train_loader, val_loader, train_t, train_y, test_t, test_y


# ════════════════════════════════════════════════════════════════════════
# Kailash-ML Engine Setup
# ════════════════════════════════════════════════════════════════════════
async def _setup_engines_async() -> (
    tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None, bool]
):
    """Initialise ExperimentTracker (kailash-ml 1.1.1 factory) + ModelRegistry."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_transformers.db"
    registry_db = "sqlite:///mlfp05_transformers_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_transformers", registry, True


def setup_engines() -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
    OnnxBridge,
]:
    """Set up all kailash-ml engines (sync wrapper).

    Returns: (conn, tracker, exp_name, registry, has_registry, bridge)
    """
    conn, tracker, exp_name, registry, has_registry = asyncio.run(
        _setup_engines_async()
    )
    bridge = OnnxBridge()
    return conn, tracker, exp_name, registry, has_registry, bridge


# ════════════════════════════════════════════════════════════════════════
# Training Utilities
# ════════════════════════════════════════════════════════════════════════
async def train_model_async(
    model: nn.Module,
    model_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    tracker: ExperimentTracker,
    exp_name: str,
    epochs: int = EPOCHS_SCRATCH,
    lr: float = 2e-3,
) -> tuple[list[float], list[float]]:
    """Train a PyTorch classifier and log every epoch to ExperimentTracker.

    Uses the modern ``tracker.track(...)`` async context manager -- bulk
    param logging, per-step metrics, automatic COMPLETED/FAILED state.
    """
    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    train_losses: list[float] = []
    val_accs: list[float] = []
    best_acc = 0.0
    best_state: dict | None = None
    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

    async with tracker.track(experiment=exp_name, run_name=model_name) as run:
        await run.log_params(
            {
                "model_type": model_name,
                "epochs": str(epochs),
                "lr": str(lr),
                "dataset_size": str(len(train_loader.dataset)),
                "batch_size": str(train_loader.batch_size),
                "trainable_params": str(param_count),
            }
        )

        for epoch in range(epochs):
            model.train()
            batch_losses = []
            for xb, yb in train_loader:
                optimizer.zero_grad()
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                batch_losses.append(loss.item())
            scheduler.step()
            epoch_loss = float(np.mean(batch_losses))
            train_losses.append(epoch_loss)

            model.eval()
            with torch.no_grad():
                correct = 0
                total = 0
                for xb, yb in val_loader:
                    preds = model(xb).argmax(dim=-1)
                    correct += int((preds == yb).sum().item())
                    total += int(yb.size(0))
                acc = correct / total
                val_accs.append(acc)

            await run.log_metrics(
                {"train_loss": epoch_loss, "val_accuracy": acc},
                step=epoch + 1,
            )
            if acc > best_acc:
                best_acc = acc
                best_state = {
                    k: v.detach().clone() for k, v in model.state_dict().items()
                }
            print(
                f"  [{model_name}] epoch {epoch+1}/{epochs}  "
                f"loss={epoch_loss:.4f}  val_acc={acc:.3f}"
            )

        await run.log_metrics(
            {
                "best_val_accuracy": best_acc,
                "final_train_loss": train_losses[-1],
            }
        )

    if best_state is not None:
        model.load_state_dict(best_state)

    return train_losses, val_accs


def train_model(
    model: nn.Module,
    model_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    tracker: ExperimentTracker,
    exp_name: str,
    epochs: int = EPOCHS_SCRATCH,
    lr: float = 2e-3,
) -> tuple[list[float], list[float]]:
    """Sync wrapper -- one asyncio.run per training call."""
    return asyncio.run(
        train_model_async(
            model, model_name, train_loader, val_loader, tracker, exp_name, epochs, lr
        )
    )


# ════════════════════════════════════════════════════════════════════════
# Visualisation Helpers
# ════════════════════════════════════════════════════════════════════════
def create_attention_heatmap(
    attn_weights: np.ndarray,
    labels: list[str],
    title: str = "Self-Attention Heatmap",
    max_tokens: int = 15,
) -> go.Figure:
    """Create a plotly heatmap from an attention weight matrix.

    Args:
        attn_weights: 2D numpy array of shape (seq, seq)
        labels: token labels for axes
        title: chart title
        max_tokens: limit display to first N non-pad tokens

    Returns:
        plotly Figure
    """
    show_len = min(max_tokens, len([w for w in labels if w != "<pad>"]))
    fig = go.Figure(
        data=go.Heatmap(
            z=attn_weights[:show_len, :show_len],
            x=labels[:show_len],
            y=labels[:show_len],
            colorscale="Viridis",
            colorbar={"title": "Attention weight"},
        )
    )
    fig.update_layout(
        title=title,
        xaxis_title="Key position",
        yaxis_title="Query position",
        width=600,
        height=500,
    )
    return fig


def get_viz() -> ModelVisualizer:
    """Return a ModelVisualizer instance."""
    return ModelVisualizer()




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 4.1: Self-Attention from Scratch
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this exercise, you will be able to:
#   - Explain why RNNs struggle with long sequences (information bottleneck)
#   - Derive scaled dot-product attention step by step using torch.einsum
#   - Explain why we divide by sqrt(d_k) (softmax saturation on large dims)
#   - Visualise attention weight matrices as heatmaps
#   - Apply attention-weighted representations to a real business problem
#
# PREREQUISITES: M5/ex_3 (RNNs, sequence modelling, nn.Module training).
# ESTIMATED TIME: ~25 min
# DATASET: AG News — 120,000 real news headlines, 4 classes
#          (World / Sports / Business / Sci-Tech).
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F


print(f"Using device: {DEVICE}")



## THEORY — Why RNNs Struggle with Long Sequences


In [ ]:
# Recurrent neural networks process tokens one at a time. Each token's
# representation depends on every previous token -- but that information
# must squeeze through a fixed-size hidden state vector. This creates
# two problems:
#
# 1. INFORMATION BOTTLENECK: By token 50, the hidden state has "forgotten"
#    the details of token 1. The model struggles to connect related words
#    that are far apart (e.g., "Singapore" at position 2 and "economy" at
#    position 30 in a news headline).
#
# 2. SEQUENTIAL COMPUTATION: RNNs process tokens one after another --
#    O(n) sequential steps. This means training cannot be parallelised
#    across sequence positions, making RNNs slow on long documents.
#
# Attention solves both problems. Instead of squeezing everything through
# a hidden state, attention lets each token directly look at every other
# token in a single step. This is the core insight of "Attention Is All
# You Need" (Vaswani et al., 2017): you don't need recurrence at all.
#
# For a single attention head:
#   Attention(Q, K, V) = softmax( Q K^T / sqrt(d_k) ) V
#
# Q (query): "what am I looking for?"   -- shape (B, L_q, d_k)
# K (key):   "what can I offer?"        -- shape (B, L_k, d_k)
# V (value): "what do I actually pass"  -- shape (B, L_k, d_v)
#
# The division by sqrt(d_k) prevents the dot products from growing with
# the embedding dimension, which would push softmax into saturation and
# kill gradients for all non-maximal keys.



## TASK 1 — Load AG News and build vocabulary


In [ ]:
train_df, test_df = load_ag_news()
vocab = build_vocab(train_df["text"].to_list())
print(f"  vocab size: {len(vocab)} (cap 15000, seq_len {MAX_LEN})")



### Checkpoint 1


In [ ]:
assert len(train_df) >= 100000, (
    f"Expected full AG News train (~120K), got {len(train_df):,}. "
    "Use the complete dataset for meaningful model comparison."
)
assert len(test_df) >= 5000, f"Expected full AG News test (~7.6K), got {len(test_df):,}"
print("\n--- Checkpoint 1 passed --- AG News loaded, vocabulary built\n")



## TASK 2 — Build: Scaled dot-product attention from scratch


Args:
        q: Query tensor of shape (B, L_q, d_k)
        k: Key tensor of shape (B, L_k, d_k)
        v: Value tensor of shape (B, L_k, d_v)
        mask: Optional mask of shape (B, L_q, L_k). Positions with 0 are
              masked out (set to -inf before softmax).

    Returns:
        (output, attention_weights) where output has shape (B, L_q, d_v)
        and attention_weights has shape (B, L_q, L_k).


In [ ]:
# We derive the attention function step by step. The key insight is that
# torch.einsum makes the batched matrix multiplication explicit and
# readable. No magic -- just linear algebra.
def scaled_dot_product_attention(
    q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, mask: torch.Tensor | None = None
) -> tuple[torch.Tensor, torch.Tensor]:
    d_k = q.size(-1)

    # Step 1: Compute raw scores via batched matrix multiplication.
    # TODO: Use torch.einsum "bqd,bkd->bqk" to compute Q*K^T scores
    # Hint: scores = torch.einsum("bqd,bkd->bqk", q, k)
    scores = ...  # YOUR CODE HERE

    # Step 2: Scale by 1/sqrt(d_k). Without this, the dot products grow
    # proportionally to d_k, pushing softmax into regions where the gradient
    # is nearly zero.
    # TODO: Divide scores by math.sqrt(d_k)
    scores = ...  # YOUR CODE HERE

    # Step 3: Apply mask (if provided). Setting masked positions to -inf
    # ensures they get zero probability after softmax.
    # TODO: Use scores.masked_fill(mask == 0, float("-inf")) when mask is not None
    if mask is not None:
        ...  # YOUR CODE HERE

    # Step 4: Softmax over the key dimension (dim=-1).
    # TODO: Apply F.softmax to get attention weights — F.softmax(scores, dim=-1)
    weights = ...  # YOUR CODE HERE

    # Step 5: Weighted sum of values using einsum.
    # TODO: Use torch.einsum "bqk,bkd->bqd" to compute weighted values
    # Hint: out = torch.einsum("bqk,bkd->bqd", weights, v)
    out = ...  # YOUR CODE HERE

    return out, weights


# Sanity check: with scaled identity-style queries, each query should
# attend most strongly to its own key position.
q_demo = torch.eye(4, 8).unsqueeze(0) * 3.0
k_demo = q_demo.clone()
v_demo = torch.arange(4 * 8, dtype=torch.float32).reshape(1, 4, 8)
_, attn_demo = scaled_dot_product_attention(q_demo, k_demo, v_demo)
print("Demo attention weights (should peak on the diagonal):")
print(attn_demo.squeeze(0).round(decimals=2))



### Checkpoint 2


In [ ]:
assert attn_demo.shape == (1, 4, 4), "Attention should be (1, 4, 4)"
diag_vals = torch.diag(attn_demo.squeeze(0))
assert diag_vals.min() > 0.5, "Diagonal should dominate (each query attends to itself)"
# INTERPRETATION: The attention matrix shows which queries attend to which
# keys. A strong diagonal means each position attends to itself -- exactly
# what we expect when Q and K are scaled identity matrices. In real text,
# the pattern is more interesting: "economy" might attend to "Singapore"
# and "growth" more than to "the" or "a".
print("\n--- Checkpoint 2 passed --- scaled dot-product attention works\n")



## TASK 3 — Visualise: Attention weight heatmap on example sentence


In [ ]:
# The attention heatmap is the transformer's "explanation" -- it reveals
# which words the model considers relevant to each other. This is not
# just a debugging tool; in production, attention visualisation helps
# domain experts validate that the model attends to the right features.
print("\n== Visualising attention on a real headline ==")

# Pick a sample headline and compute attention through a learned projection
sample_text = test_df["text"].to_list()[0]
sample_words = sample_text.lower().split()[:MAX_LEN]
sample_indices = torch.tensor(
    [text_to_indices(sample_text, vocab, MAX_LEN)], dtype=torch.long
)

# TODO: Create embedding layer and compute self-attention on a real headline
# Hint: embed_dim = 32; embedding = torch.nn.Embedding(len(vocab), embed_dim, padding_idx=0)
# Then: embedded = embedding(sample_indices) inside torch.no_grad()
# Then: call scaled_dot_product_attention(embedded, embedded, embedded)
embed_dim = 32
embedding = (
    ...
)  # YOUR CODE HERE — torch.nn.Embedding(len(vocab), embed_dim, padding_idx=0)
with torch.no_grad():
    embedded = ...  # YOUR CODE HERE — embedding(sample_indices)
    _, sample_attn = (
        ...
    )  # YOUR CODE HERE — scaled_dot_product_attention(embedded, embedded, embedded)
    sample_attn_np = sample_attn[0].numpy()  # (MAX_LEN, MAX_LEN)

# Build word labels for the heatmap
word_labels = sample_words + ["<pad>"] * (MAX_LEN - len(sample_words))

fig_attn = create_attention_heatmap(
    sample_attn_np,
    word_labels,
    title=f"Self-Attention on: '{sample_text[:60]}...'",
    max_tokens=15,
)
fig_attn.write_html("ex_4_1_attention_heatmap.html")
print(f"  Headline: {sample_text[:80]!r}")
print(f"  Attention heatmap saved to ex_4_1_attention_heatmap.html")
print(f"  True label: {CLASS_NAMES[test_df['label'].to_list()[0]]}")



### Checkpoint 3


In [ ]:
assert sample_attn_np.shape == (
    MAX_LEN,
    MAX_LEN,
), "Attention should cover full sequence"
assert Path(
    "ex_4_1_attention_heatmap.html"
).exists(), "Attention heatmap should be saved"
# INTERPRETATION: The heatmap shows which words attend to which. Even with
# random embeddings, you can see structural patterns: content words attend
# to other content words, and padding positions form their own cluster.
# With trained embeddings, these patterns become meaningful -- "Singapore"
# would strongly attend to "economy", "growth", and "GDP".
print("\n--- Checkpoint 3 passed --- attention heatmap visualised\n")



## TASK 4 — Apply: Document Similarity for a Singapore Law Firm


This is the core of attention-based similarity: instead of averaging
    all word embeddings equally, attention lets important words (determined
    by their relationship to all other words) contribute more.


In [ ]:
# SCENARIO: Rajah & Tann, one of Singapore's largest law firms, processes
# thousands of legal documents monthly. Lawyers need to find prior case
# precedents that are relevant to their current case. Traditional keyword
# search misses semantic connections (e.g., "breach of fiduciary duty" is
# related to "director's negligence" even though they share no keywords).
#
# BUSINESS VALUE: Attention-weighted document representations capture
# semantic similarity that keyword matching cannot. A lawyer searching
# for "breach of fiduciary duty" finds related cases about director
# negligence, trustee misconduct, and corporate governance failures --
# reducing precedent research from 4-6 hours to 15-30 minutes per case.
#
# DOLLAR IMPACT: At S$500-800/hour for senior associates, saving 3-5
# hours per case on a firm handling ~200 commercial litigation cases/year
# translates to S$300K-800K in recovered associate time annually.
print("\n== Application: Document Similarity for Rajah & Tann (Singapore Law) ==")

query_texts = [
    "Wall Street stocks fall as economy shows signs of weakness",
    "Technology companies report record profits in quarterly earnings",
    "Olympic athletes break world records in swimming competition",
]
query_labels = ["Business", "Sci/Tech", "Sports"]

embedding_legal = torch.nn.Embedding(len(vocab), embed_dim, padding_idx=0)

candidate_texts = test_df["text"].to_list()[:50]
candidate_labels = [CLASS_NAMES[l] for l in test_df["label"].to_list()[:50]]


def get_attention_representation(text: str) -> torch.Tensor:
    indices = torch.tensor([text_to_indices(text, vocab, MAX_LEN)], dtype=torch.long)
    with torch.no_grad():
        # TODO: Compute embedding, apply self-attention, mean-pool over non-pad positions
        # Step 1: emb = embedding_legal(indices)
        # Step 2: attn_out, _ = scaled_dot_product_attention(emb, emb, emb)
        # Step 3: mask = (indices != 0).float().unsqueeze(-1)
        # Step 4: pooled = (attn_out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        emb = ...  # YOUR CODE HERE
        attn_out, _ = ...  # YOUR CODE HERE
        mask = ...  # YOUR CODE HERE — (indices != 0).float().unsqueeze(-1)
        pooled = (
            ...
        )  # YOUR CODE HERE — (attn_out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    return pooled.squeeze(0)  # (embed_dim,)


# Compute representations for all candidates
candidate_reps = torch.stack([get_attention_representation(t) for t in candidate_texts])

print(f"\n  Query documents: {len(query_texts)}")
print(f"  Candidate pool: {len(candidate_texts)} headlines")

# TODO: For each query, compute its representation, find top-3 similar candidates
# Hint: q_rep = get_attention_representation(q_text)
# Hint: similarities = F.cosine_similarity(q_rep.unsqueeze(0), candidate_reps, dim=1)
# Hint: top_k = similarities.topk(3)
for qi, (q_text, q_label) in enumerate(zip(query_texts, query_labels)):
    q_rep = ...  # YOUR CODE HERE — get_attention_representation(q_text)
    similarities = (
        ...
    )  # YOUR CODE HERE — F.cosine_similarity(q_rep.unsqueeze(0), candidate_reps, dim=1)
    top_k = ...  # YOUR CODE HERE — similarities.topk(3)

    print(f"\n  Query ({q_label}): '{q_text[:60]}'")
    for rank, (score, idx) in enumerate(
        zip(top_k.values.tolist(), top_k.indices.tolist())
    ):
        match_label = candidate_labels[idx]
        match_text = candidate_texts[idx][:60]
        marker = "[correct]" if match_label == q_label else "[cross-topic]"
        print(f"    Top-{rank+1} (sim={score:.3f}) {marker}: '{match_text}'")



### Checkpoint 4


In [ ]:
assert candidate_reps.shape == (
    50,
    embed_dim,
), "Should have 50 candidate representations"
# INTERPRETATION: Even with untrained embeddings, the attention mechanism
# captures structural patterns in text that aid similarity search. With
# trained embeddings (as in the Transformer and BERT exercises that follow),
# the similarity becomes semantically meaningful -- "breach of fiduciary duty"
# would cluster with "director's negligence" in the attention-weighted space.
#
# BUSINESS IMPACT for Rajah & Tann:
#   - 200 commercial litigation cases/year
#   - 3-5 hours saved per case on precedent research
#   - At S$500-800/hour senior associate rate
#   - Annual saving: S$300K-800K in recovered associate time
#   - Additional value: fewer missed precedents -> better case outcomes
print("\n--- Checkpoint 4 passed --- Singapore law firm application complete\n")



## REFLECTION


[x] Understood WHY RNNs struggle (information bottleneck, O(n) sequential)
  [x] Derived scaled dot-product attention step by step
  [x] Explained the 1/sqrt(d_k) scaling factor (prevents softmax saturation)
  [x] Visualised attention weights as an interpretable heatmap
  [x] Applied attention-weighted representations to document similarity
  [x] Evaluated business impact for Singapore legal industry

  KEY INSIGHT:
    Attention lets every token directly access every other token in one
    step. No information bottleneck. No sequential processing. This is
    the foundation that makes Transformers and BERT possible.

  Next: In 02_transformer_encoder.py, you'll build multi-head attention
  and a full Transformer encoder classifier that uses this attention
  mechanism with multiple parallel "heads" for richer representations.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — Self-Attention from Scratch")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — none for this file (inference-only derivation)


In [ ]:
# This exercise derives scaled dot-product attention in NumPy-style
# torch operations. There is no training loop and no learned weights
# (only untrained embeddings used to populate demo heatmaps), so the
# five diagnostic instruments (Stethoscope / Blood Test / X-Ray /
# Vital Signs / Prescription Pad) do not apply here.
#
# STUDENT INTERPRETATION GUIDE: the attention heatmap ITSELF is the
# diagnostic output for this file. Read it the way you would read an
# X-Ray in ex_4/02 — "which positions light up?" The demo heatmap
# above should show a strong diagonal (each position attends to
# itself) because Q = K = scaled identity. In ex_4/02 the trained
# Transformer's heatmap will show OFF-DIAGONAL structure — content
# words attending to related content words. That is the "attention
# has learned something" signal. If the trained model's heatmap
# stays diagonal, the attention heads have collapsed (Prescription
# Pad row: "attention collapse — add dropout, increase d_model, or
# reduce n_heads").
#
# CONNECT TO SLIDE 5.4: The slide calls attention "a soft, learned
# version of a look-up table". The diagonal demo heatmap is the
# "hard look-up" (each query finds exactly its matching key); the
# trained Transformer's heatmap is the "soft, learned" version.



## REFLECTION


[x] Understood WHY RNNs struggle (information bottleneck, O(n) sequential)
  [x] Derived scaled dot-product attention step by step
  [x] Explained the 1/sqrt(d_k) scaling factor (prevents softmax saturation)
  [x] Visualised attention weights as an interpretable heatmap
  [x] Applied attention-weighted representations to document similarity
  [x] Evaluated business impact for Singapore legal industry

  KEY INSIGHT:
    Attention lets every token directly access every other token in one
    step. No information bottleneck. No sequential processing. This is
    the foundation that makes Transformers and BERT possible.

  Next: In 02_transformer_encoder.py, you'll build multi-head attention
  and a full Transformer encoder classifier that uses this attention
  mechanism with multiple parallel "heads" for richer representations.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — Self-Attention from Scratch")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — none for this file (inference-only derivation)


In [ ]:
# This exercise derives scaled dot-product attention in NumPy-style
# torch operations. There is no training loop and no learned weights
# (only untrained embeddings used to populate demo heatmaps), so the
# five diagnostic instruments (Stethoscope / Blood Test / X-Ray /
# Vital Signs / Prescription Pad) do not apply here.
#
# STUDENT INTERPRETATION GUIDE: the attention heatmap ITSELF is the
# diagnostic output for this file. Read it the way you would read an
# X-Ray in ex_4/02 — "which positions light up?" The demo heatmap
# above should show a strong diagonal (each position attends to
# itself) because Q = K = scaled identity. In ex_4/02 the trained
# Transformer's heatmap will show OFF-DIAGONAL structure — content
# words attending to related content words. That is the "attention
# has learned something" signal. If the trained model's heatmap
# stays diagonal, the attention heads have collapsed (Prescription
# Pad row: "attention collapse — add dropout, increase d_model, or
# reduce n_heads").
#
# CONNECT TO SLIDE 5.4: The slide calls attention "a soft, learned
# version of a look-up table". The diagonal demo heatmap is the
# "hard look-up" (each query finds exactly its matching key); the
# trained Transformer's heatmap is the "soft, learned" version.



## REFLECTION


[x] Understood WHY RNNs struggle (information bottleneck, O(n) sequential)
  [x] Derived scaled dot-product attention step by step
  [x] Explained the 1/sqrt(d_k) scaling factor (prevents softmax saturation)
  [x] Visualised attention weights as an interpretable heatmap
  [x] Applied attention-weighted representations to document similarity
  [x] Evaluated business impact for Singapore legal industry

  KEY INSIGHT:
    Attention lets every token directly access every other token in one
    step. No information bottleneck. No sequential processing. This is
    the foundation that makes Transformers and BERT possible.

  Next: In 02_transformer_encoder.py, you'll build multi-head attention
  and a full Transformer encoder classifier that uses this attention
  mechanism with multiple parallel "heads" for richer representations.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — Self-Attention from Scratch")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — none for this file (inference-only derivation)


In [ ]:
# This exercise derives scaled dot-product attention in NumPy-style
# torch operations. There is no training loop and no learned weights
# (only untrained embeddings used to populate demo heatmaps), so the
# five diagnostic instruments (Stethoscope / Blood Test / X-Ray /
# Vital Signs / Prescription Pad) do not apply here.
#
# STUDENT INTERPRETATION GUIDE: the attention heatmap ITSELF is the
# diagnostic output for this file. Read it the way you would read an
# X-Ray in ex_4/02 — "which positions light up?" The demo heatmap
# above should show a strong diagonal (each position attends to
# itself) because Q = K = scaled identity. In ex_4/02 the trained
# Transformer's heatmap will show OFF-DIAGONAL structure — content
# words attending to related content words. That is the "attention
# has learned something" signal. If the trained model's heatmap
# stays diagonal, the attention heads have collapsed (Prescription
# Pad row: "attention collapse — add dropout, increase d_model, or
# reduce n_heads").
#
# CONNECT TO SLIDE 5.4: The slide calls attention "a soft, learned
# version of a look-up table". The diagonal demo heatmap is the
# "hard look-up" (each query finds exactly its matching key); the
# trained Transformer's heatmap is the "soft, learned" version.



## REFLECTION


[x] Understood WHY RNNs struggle (information bottleneck, O(n) sequential)
  [x] Derived scaled dot-product attention step by step
  [x] Explained the 1/sqrt(d_k) scaling factor (prevents softmax saturation)
  [x] Visualised attention weights as an interpretable heatmap
  [x] Applied attention-weighted representations to document similarity
  [x] Evaluated business impact for Singapore legal industry

  KEY INSIGHT:
    Attention lets every token directly access every other token in one
    step. No information bottleneck. No sequential processing. This is
    the foundation that makes Transformers and BERT possible.

  Next: In 02_transformer_encoder.py, you'll build multi-head attention
  and a full Transformer encoder classifier that uses this attention
  mechanism with multiple parallel "heads" for richer representations.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — Self-Attention from Scratch")
print("=" * 70)
print(
)

